In [2]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import timedelta

fake = Faker()
Faker.seed(42) # For reproducibility
np.random.seed(42)

# --- CONFIGURATION ---
NUM_CUSTOMERS = 300
NUM_REPS = 15
NUM_PRODUCTS = 40
NUM_LEADS = 1500 # More leads than orders (realistic conversion rate)

# 1. GENERATE BASE TABLES
customers = pd.DataFrame({
    'customer_id': range(1, NUM_CUSTOMERS + 1),
    'company_name': [fake.company() for _ in range(NUM_CUSTOMERS)],
    'industry': [random.choice(['Tech', 'Manufacturing', 'Healthcare', 'Finance', 'Retail']) for _ in range(NUM_CUSTOMERS)],
    'region': [random.choice(['North', 'South', 'East', 'West']) for _ in range(NUM_CUSTOMERS)],
    'acquisition_date': [fake.date_between(start_date='-3y', end_date='today') for _ in range(NUM_CUSTOMERS)]
})

reps = pd.DataFrame({
    'rep_id': range(1, NUM_REPS + 1),
    'rep_name': [fake.name() for _ in range(NUM_REPS)],
    'region': [random.choice(['North', 'South', 'East', 'West']) for _ in range(NUM_REPS)],
    'hire_date': [fake.date_between(start_date='-5y', end_date='-1y') for _ in range(NUM_REPS)]
})

products = pd.DataFrame({
    'product_id': range(1, NUM_PRODUCTS + 1),
    'product_name': [fake.catch_phrase() for _ in range(NUM_PRODUCTS)],
    'category': [random.choice(['Hardware', 'Software', 'Services', 'Maintenance']) for _ in range(NUM_PRODUCTS)],
    'unit_cost': [round(random.uniform(50, 500), 2) for _ in range(NUM_PRODUCTS)],
    'unit_price': [round(random.uniform(100, 1000), 2) for _ in range(NUM_PRODUCTS)]
})
# Ensure price is always higher than cost
products['unit_price'] = products.apply(lambda row: max(row['unit_price'], row['unit_cost'] * 1.2), axis=1)

# 2. GENERATE LEADS (CRM)
leads_data = []
for i in range(1, NUM_LEADS + 1):
    created = fake.date_between(start_date='-1y', end_date='today')
    status = random.choices(['Won', 'Lost', 'Open'], weights=[40, 40, 20])[0] # 40% win rate

    if status == 'Won':
        closed = created + timedelta(days=random.randint(5, 90))
        stage = 'Closed Won'
    elif status == 'Lost':
        closed = created + timedelta(days=random.randint(5, 60))
        stage = random.choice(['Lost', 'Disqualified'])
    else: # Open
        closed = None
        stage = random.choice(['Prospecting', 'Negotiation', 'Proposal'])
        # Inject "Stuck" deals: 10% of open deals haven't moved in 100+ days
        if random.random() < 0.10:
            created = fake.date_between(start_date='-1y', end_date='-100d')

    leads_data.append({
        'lead_id': i,
        'customer_id': random.randint(1, NUM_CUSTOMERS),
        'rep_id': random.randint(1, NUM_REPS),
        'lead_source': random.choice(['Website', 'Referral', 'Cold_Call', 'LinkedIn']),
        'stage': stage,
        'status': status,
        'created_date': created,
        'closed_date': closed
    })
leads = pd.DataFrame(leads_data)

# 3. GENERATE ORDERS & INVOICES (ERP)
# Only 'Won' leads become orders
won_leads = leads[leads['status'] == 'Won'].copy()

orders_data = []
invoices_data = []
order_id = 1
invoice_id = 1

for _, lead in won_leads.iterrows():
    # A customer might buy multiple products in one order, but let's keep it 1 product per order for simplicity
    prod_id = random.randint(1, NUM_PRODUCTS)
    qty = random.randint(1, 10)
    price = products.loc[products['product_id'] == prod_id, 'unit_price'].values[0]

    order_date = lead['closed_date'] + timedelta(days=random.randint(1, 5))

    # INJECTING MESSINESS: 5% of orders ship AFTER they are invoiced (Process error)
    if random.random() < 0.05:
        ship_date = order_date + timedelta(days=random.randint(15, 30))
        invoice_date = order_date + timedelta(days=random.randint(1, 5))
    else:
        ship_date = order_date + timedelta(days=random.randint(1, 10))
        invoice_date = ship_date + timedelta(days=random.randint(1, 3))

    orders_data.append({
        'order_id': order_id,
        'lead_id': lead['lead_id'],
        'product_id': prod_id,
        'quantity': qty,
        'order_date': order_date,
        'ship_date': ship_date,
        'total_amount': round(qty * price, 2)
    })

    # Generate Invoice
    due_date = invoice_date + timedelta(days=30) # Net 30 terms

    # INJECTING MESSINESS: 15% of invoices are unpaid
    is_paid = random.random() > 0.15
    payment_date = due_date - timedelta(days=random.randint(0, 15)) if is_paid else None

    status = 'Paid' if is_paid else ('Overdue' if fake.date_object() > due_date else 'Unpaid')

    invoices_data.append({
        'invoice_id': invoice_id,
        'order_id': order_id,
        'invoice_date': invoice_date,
        'due_date': due_date,
        'payment_date': payment_date,
        'status': status
    })

    order_id += 1
    invoice_id += 1

orders = pd.DataFrame(orders_data)
invoices = pd.DataFrame(invoices_data)

# 4. EXPORT TO CSV
customers.to_csv('customers.csv', index=False)
reps.to_csv('sales_reps.csv', index=False)
products.to_csv('products.csv', index=False)
leads.to_csv('leads.csv', index=False)
orders.to_csv('orders.csv', index=False)
invoices.to_csv('invoices.csv', index=False)

print("Data generation complete! 6 CSV files created.")

Data generation complete! 6 CSV files created.
